# Network 멀티 에이전트 — 조사 → 차트

**Network(네트워크)** 구조는 에이전트들이 **서로 직접 핸드오프** 하며 협업한다 (중앙 관리자 없음). 여기서는 두 전문가가 작업을 주고받는다:
- **researcher**: 웹검색으로 데이터를 조사
- **chart_generator**: 파이썬 코드를 실행해 차트 생성

```
START → researcher ⇄ chart_generator → (누군가 'FINAL ANSWER' 하면) END
```

각 노드는 [basics] `Command(goto=...)` 로 상대에게 넘기거나, 끝났으면 END 로 간다.

> `ANTHROPIC_API_KEY`, `TAVILY_API_KEY` 필요.

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
for k in ["ANTHROPIC_API_KEY", "TAVILY_API_KEY"]:
    assert os.environ.get(k), f"{k} 가 .env 에 없습니다"
print("환경변수 로드 완료")

## 공용 시스템 프롬프트

두 에이전트가 공유하는 협업 지침. 핵심: **자기가 할 수 있는 만큼만 하고**, 최종 결과가 나오면 응답 앞에 `FINAL ANSWER` 를 붙여 팀이 멈추도록 신호한다.

In [ ]:
def make_system_prompt(suffix: str) -> str:
    return (
        "You are a helpful AI assistant, collaborating with other assistants."
        " Use the provided tools to progress towards answering the question."
        " If you are unable to fully answer, that's OK, another assistant with different tools"
        " will help where you left off. Execute what you can to make progress."
        " If you or any of the other assistants have the final answer or deliverable,"
        " prefix your response with FINAL ANSWER so the team knows to stop."
        f"\n{suffix}"
    )

## 1. researcher — 웹검색 조사 에이전트
[basics] `create_react_agent` 에 Tavily 검색 도구를 준다.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_anthropic import ChatAnthropic
from langgraph.prebuilt import create_react_agent

tavily_tool = TavilySearchResults(max_results=5)
llm = ChatAnthropic(model="claude-3-7-sonnet-20250219")

research_agent = create_react_agent(
    llm,
    tools=[tavily_tool],
    prompt=make_system_prompt(
        "You can only do research. You are working with a chart generator colleague."
    ),
)

### researcher 노드 — 핸드오프 로직

[basics] 에이전트를 호출하고, 응답에 `FINAL ANSWER` 가 있으면 END, 아니면 `chart_generator` 로 넘긴다. 메시지에 `name` 을 달아 누가 한 말인지 표시한다.

In [ ]:
from typing import Literal
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import MessagesState, END
from langgraph.types import Command

def get_next_node(last_message: BaseMessage, goto: str):
    # 누군가 FINAL ANSWER 를 내면 팀 종료
    if "FINAL ANSWER" in last_message.content:
        return END
    return goto

def research_node(state: MessagesState) -> Command[Literal["chart_generator", END]]:
    result = research_agent.invoke(state)
    goto = get_next_node(result["messages"][-1], "chart_generator")
    # 마지막 메시지를 'researcher' 가 한 말로 표시
    result["messages"][-1] = HumanMessage(
        content=result["messages"][-1].content, name="researcher"
    )
    return Command(update={"messages": result["messages"]}, goto=goto)

## 2. chart_generator — 차트 생성 에이전트

파이썬 코드를 실행하는 도구를 준다. 조사 결과를 받아 matplotlib 등으로 차트를 그린다.

> ⚠️ `exec` 로 임의 코드를 실행하므로 학습용으로만. 실제로는 샌드박스 필요.

In [ ]:
from typing import Annotated
from langchain_core.tools import tool

@tool
def python_exec_tool(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Use this to execute python code. Print values you want to see with print(...)."""
    try:
        result = exec(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return (
        f"Successfully executed:\n{code}\nStdout: {result}\n\n"
        "If you have completed all tasks, respond with FINAL ANSWER."
    )

chart_agent = create_react_agent(
    llm,
    [python_exec_tool],
    prompt=make_system_prompt(
        "You can only generate charts. You are working with a researcher colleague."
    ),
)

In [ ]:
def chart_node(state: MessagesState) -> Command[Literal["researcher", END]]:
    result = chart_agent.invoke(state)
    goto = get_next_node(result["messages"][-1], "researcher")
    result["messages"][-1] = HumanMessage(
        content=result["messages"][-1].content, name="chart_generator"
    )
    return Command(update={"messages": result["messages"]}, goto=goto)

## 3. 그래프 조립

[basics] 두 노드만 등록한다. researcher ⇄ chart_generator 의 왕복은 각 노드의 `Command(goto=...)` 가 만들어주므로 별도 엣지가 거의 없다.

In [ ]:
from langgraph.graph import StateGraph, START

graph_builder = StateGraph(MessagesState)
graph_builder.add_node("researcher", research_node)
graph_builder.add_node("chart_generator", chart_node)
graph_builder.add_edge(START, "researcher")
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트

조사 → 차트 협업이 일어난다. `recursion_limit` 을 넉넉히 준다 (왕복이 여러 번 일어날 수 있음).

In [ ]:
events = graph.stream(
    {"messages": [(
        "user",
        "Research South Korea's GDP for 2020-2024, then draw a line chart. "
        "Use English text in the chart. Stop after the chart is made.",
    )]},
    {"recursion_limit": 150},
)
for s in events:
    print(s)
    print("----")

## 정리

- **Network** = 에이전트들이 서로 직접 핸드오프 (중앙 관리자 없음)
- 각 노드가 `Command(goto=상대 or END)` 로 다음 차례를 결정
- 공용 프롬프트의 **`FINAL ANSWER`** 신호로 협업 종료를 판단
- 메시지 `name` 으로 발화 주체를 구분

다음: 관리자(supervisor)가 작업을 배분하는 **Supervisor** 구조.